**NYC Open Data**

311 Service Requests in Manhattan from First Quarter of 2026 (1/24/2026 - 4/24/2026)

In [ ]:
#import libraries
import pandas as pd


In [ ]:
#mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#get the data
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NYC_311_Manhattan_2026/nyc_311_manhattan_2026_q1.csv')

/tmp/ipykernel_16880/2862941338.py:2: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NYC_311_Manhattan_2026/nyc_311_manhattan_2026_q1.csv')


In [ ]:
#keep only the needed columns for analysis
columns_to_keep = [
    'Created Date',
    'Closed Date',
    'Agency',
    'Problem (formerly Complaint Type)',
    'Problem Detail (formerly Descriptor)',
    'Location Type',
    'Incident Zip',
    'Status',
    'Resolution Description',
    'Borough',
    'Latitude',
    'Longitude',
]

df = df[columns_to_keep]

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197922 entries, 0 to 197921
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Created Date                          197922 non-null  object 
 1   Closed Date                           172952 non-null  object 
 2   Agency                                197922 non-null  object 
 3   Problem (formerly Complaint Type)     197922 non-null  object 
 4   Problem Detail (formerly Descriptor)  190587 non-null  object 
 5   Location Type                         167663 non-null  object 
 6   Incident Zip                          195656 non-null  float64
 7   Status                                197922 non-null  object 
 8   Resolution Description                188345 non-null  object 
 9   Borough                               197922 non-null  object 
 10  Latitude                              193646 non-null  float64
 11  

In [ ]:
df.shape[0]

197922

In [ ]:
df.head()

,Created Date,Closed Date,Agency,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Location Type,Incident Zip,Status,Resolution Description,Borough,Latitude,Longitude
0,04/24/2026 01:45:44 AM,NaN,DOHMH,Food Establishment,Rodents/Insects/Garbage,Restaurant/Bar/Deli/Bakery,10002.0,In Progress,NaN,MANHATTAN,40.711703,-73.994268
1,04/24/2026 01:45:07 AM,NaN,DOHMH,Smoking or Vaping,Allowed in Smoke Free Area,Residential Building,10036.0,In Progress,NaN,MANHATTAN,40.760864,-73.996336
2,04/24/2026 01:43:01 AM,NaN,NYPD,Illegal Parking,Blocked Sidewalk,Street/Sidewalk,10032.0,In Progress,NaN,MANHATTAN,40.839022,-73.937521
3,04/24/2026 01:42:58 AM,NaN,NYPD,Noise - Residential,Loud Music/Party,Residential Building/House,10009.0,In Progress,NaN,MANHATTAN,40.724868,-73.980742
4,04/24/2026 01:42:17 AM,NaN,NYPD,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,10027.0,In Progress,NaN,MANHATTAN,40.814140,-73.959434


In [ ]:
df.isnull()


,Created Date,Closed Date,Agency,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Location Type,Incident Zip,Status,Resolution Description,Borough,Latitude,Longitude
0,False,True,False,False,False,False,False,False,True,False,False,False
1,False,True,False,False,False,False,False,False,True,False,False,False
2,False,True,False,False,False,False,False,False,True,False,False,False
3,False,True,False,False,False,False,False,False,True,False,False,False
4,False,True,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
197917,False,False,False,False,False,False,False,False,False,False,False,False
197918,False,False,False,False,False,True,False,False,False,False,False,False
197919,False,False,False,False,False,True,False,False,False,False,False,False
197920,False,False,False,False,False,False,False,False,False,False,False,False


In [ ]:
# convert Created Date to datetime
df['Created Date'] = pd.to_datetime(df['Created Date'], errors='coerce')
df['Closed Date'] = pd.to_datetime(df['Closed Date'], errors='coerce')

In [ ]:
#checking to see if columns converted to datetime
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197922 entries, 0 to 197921
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype         
---  ------                                --------------   -----         
 0   Created Date                          197922 non-null  datetime64[ns]
 1   Closed Date                           172952 non-null  datetime64[ns]
 2   Agency                                197922 non-null  object        
 3   Problem (formerly Complaint Type)     197922 non-null  object        
 4   Problem Detail (formerly Descriptor)  190587 non-null  object        
 5   Location Type                         167663 non-null  object        
 6   Incident Zip                          195656 non-null  float64       
 7   Status                                197922 non-null  object        
 8   Resolution Description                188345 non-null  object        
 9   Borough                               197922 non-null  obje

In [ ]:
#create new column to calculate resolution time using created date and closed date
df['Resolution_Time'] = df['Closed Date'] - df['Created Date']

In [ ]:
df.isnull().sum()

,0
Created Date,0
Closed Date,24970
Agency,0
Problem (formerly Complaint Type),0
Problem Detail (formerly Descriptor),7335
Location Type,30259
Incident Zip,2266
Status,0
Resolution Description,9577
Borough,0


Created Date, Agency, Problem, Status, and Borough all have 0 null values.

Columns with large amounts of null values:

Closed Date, Problem Detail, Location Type, Resolution Time, Latitude and Longitude, Resolution Description, and Incident Zip


In [ ]:
#sort Closed Date to reflect problems that are still open
df_closed = df[df['Closed Date'].notna()]

In [ ]:
#drop Location Type; it has 30259 null values
df = df.drop(columns=['Location Type'])

In [ ]:
df.isnull().sum()

,0
Created Date,0
Closed Date,24970
Agency,0
Problem (formerly Complaint Type),0
Problem Detail (formerly Descriptor),7335
Incident Zip,2266
Status,0
Resolution Description,9577
Borough,0
Latitude,4276


In [ ]:
#latitude and longitude missing null values work around
df_map = df.dropna(subset=['Latitude','Longitude'])

In [ ]:
# slice the date for more granular features for Power BI
df['year'] = df['Created Date'].dt.year
df['month'] = df['Created Date'].dt.month
df['hour'] = df['Created Date'].dt.hour

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197922 entries, 0 to 197921
Data columns (total 15 columns):
 #   Column                                Non-Null Count   Dtype          
---  ------                                --------------   -----          
 0   Created Date                          197922 non-null  datetime64[ns] 
 1   Closed Date                           172952 non-null  datetime64[ns] 
 2   Agency                                197922 non-null  object         
 3   Problem (formerly Complaint Type)     197922 non-null  object         
 4   Problem Detail (formerly Descriptor)  190587 non-null  object         
 5   Incident Zip                          195656 non-null  float64        
 6   Status                                197922 non-null  object         
 7   Resolution Description                188345 non-null  object         
 8   Borough                               197922 non-null  object         
 9   Latitude                              193646 non

In [ ]:
# make all column names lower case for easy access
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197922 entries, 0 to 197921
Data columns (total 15 columns):
 #   Column                                Non-Null Count   Dtype          
---  ------                                --------------   -----          
 0   created_date                          197922 non-null  datetime64[ns] 
 1   closed_date                           172952 non-null  datetime64[ns] 
 2   agency                                197922 non-null  object         
 3   problem_(formerly_complaint_type)     197922 non-null  object         
 4   problem_detail_(formerly_descriptor)  190587 non-null  object         
 5   incident_zip                          195656 non-null  float64        
 6   status                                197922 non-null  object         
 7   resolution_description                188345 non-null  object         
 8   borough                               197922 non-null  object         
 9   latitude                              193646 non

In [ ]:
#export a cleaned NYC dataset for use in Power BI
df.to_csv('nyc_311_cleaned_manhattan_2026.csv', index=False)

In [ ]:
# saving a Closed Date version for working with resolution time
df_closed = df[df['closed_date'].notna()]
df_closed.to_csv('nyc_311_closed_only_2026.csv', index=False)

In [ ]:
#save to Google Drive to ensure the cleaned file is saved properly
df.to_csv('/content/drive/MyDrive/Colab Notebooks/nyc_311_cleaned_manhattan_2026.csv', index=False)



In [ ]:
#save the Closed Date version too
df.to_csv('/content/drive/MyDrive/Colab Notebooks/nyc_311_closed_only_2026.csv', index=False)

In [ ]:
#final shape of the data
df.shape[0]

197922